In [2]:
!pip install PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.8 MB/s eta 0:00:00


In [4]:
!pip install python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.1 MB/s eta 0:00:00


In [6]:
import os
import json
import uuid
import logging
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime
import hashlib
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Document processing
import PyPDF2
import docx
from pathlib import Path

# Web scraping for legal updates
import requests
from bs4 import BeautifulSoup

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Download required NLTK data - FIXED VERSION
def download_nltk_data():
    """Download required NLTK data with proper error handling"""
    required_data = [
        ('tokenizers/punkt', 'punkt'),
        ('tokenizers/punkt_tab', 'punkt_tab'),
        ('corpora/stopwords', 'stopwords'),
        ('corpora/wordnet', 'wordnet'),
        ('corpora/omw-1.4', 'omw-1.4')
    ]

    for path, name in required_data:
        try:
            nltk.data.find(path)
            print(f"✓ {name} already available")
        except LookupError:
            try:
                print(f"Downloading {name}...")
                nltk.download(name, quiet=False)
                print(f"✓ {name} downloaded successfully")
            except Exception as e:
                print(f"⚠ Warning: Could not download {name}: {e}")

# Call the download function
download_nltk_data()

@dataclass
class LegalDocument:
    """Represents a legal document with metadata"""
    id: str
    title: str
    content: str
    document_type: str  # contract, regulation, case_law, statute
    jurisdiction: str
    practice_area: List[str]  # corporate, litigation, IP, etc.
    date_created: str
    key_terms: List[str]
    confidence_score: float = 0.0
    summary: str = ""

@dataclass
class LegalQuery:
    """Represents a legal query with context"""
    query: str
    practice_area: Optional[str] = None
    jurisdiction: Optional[str] = None
    urgency: str = "medium"  # low, medium, high
    query_type: str = "general"  # research, compliance, drafting, analysis

class LegalTextProcessor:
    """Advanced text processor for legal documents"""

    def __init__(self):
        try:
            self.lemmatizer = WordNetLemmatizer()
            self.stop_words = set(stopwords.words('english'))
        except Exception as e:
            print(f"Warning: NLTK data issue: {e}")
            self.lemmatizer = None
            self.stop_words = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'])

        # Legal-specific terms that shouldn't be removed
        self.legal_terms = {
            'plaintiff', 'defendant', 'contract', 'agreement', 'liability',
            'damages', 'breach', 'clause', 'provision', 'statute', 'regulation',
            'compliance', 'jurisdiction', 'precedent', 'ruling', 'judgment'
        }

        # Legal document patterns
        self.legal_patterns = {
            'case_citation': r'\d+\s+\w+\.?\s+\d+',
            'statute_ref': r'§\s*\d+(\.\d+)*',
            'contract_clause': r'Section\s+\d+(\.\d+)*',
            'dollar_amount': r'\$[\d,]+(\.\d{2})?',
            'date_pattern': r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
        }

    def extract_legal_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract legal entities from text"""
        entities = {
            'case_citations': [],
            'statute_references': [],
            'contract_clauses': [],
            'dollar_amounts': [],
            'dates': []
        }

        for entity_type, pattern in self.legal_patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            if entity_type == 'case_citation':
                entities['case_citations'].extend(matches)
            elif entity_type == 'statute_ref':
                entities['statute_references'].extend(matches)
            elif entity_type == 'contract_clause':
                entities['contract_clauses'].extend(matches)
            elif entity_type == 'dollar_amount':
                entities['dollar_amounts'].extend(matches)
            elif entity_type == 'date_pattern':
                entities['dates'].extend(matches)

        return entities

    def preprocess_legal_text(self, text: str) -> str:
        """Preprocess legal text while preserving important legal terms"""
        try:
            # Convert to lowercase but preserve legal terms
            words = word_tokenize(text.lower())
        except Exception:
            # Fallback if NLTK tokenizer fails
            words = text.lower().split()

        processed_words = []
        for word in words:
            if word.isalpha() and len(word) > 2:
                if word not in self.stop_words or word in self.legal_terms:
                    if self.lemmatizer:
                        try:
                            processed_words.append(self.lemmatizer.lemmatize(word))
                        except:
                            processed_words.append(word)
                    else:
                        processed_words.append(word)

        return ' '.join(processed_words)

    def extract_key_terms(self, text: str, max_terms: int = 10) -> List[str]:
        """Extract key legal terms from text"""
        # Use TF-IDF to find important terms
        vectorizer = TfidfVectorizer(max_features=max_terms, stop_words='english')

        try:
            tfidf_matrix = vectorizer.fit_transform([text])
            feature_names = vectorizer.get_feature_names_out()
            tfidf_scores = tfidf_matrix.toarray()[0]

            # Get top terms with their scores
            term_scores = list(zip(feature_names, tfidf_scores))
            term_scores.sort(key=lambda x: x[1], reverse=True)

            return [term for term, score in term_scores if score > 0]
        except:
            # Fallback to simple word frequency
            try:
                words = word_tokenize(text.lower())
            except:
                words = text.lower().split()

            word_freq = {}
            for word in words:
                if word.isalpha() and word not in self.stop_words:
                    word_freq[word] = word_freq.get(word, 0) + 1

            return sorted(word_freq.keys(), key=word_freq.get, reverse=True)[:max_terms]

class DocumentParser:
    """Parse various document formats"""

    @staticmethod
    def parse_pdf(file_path: str) -> str:
        """Extract text from PDF"""
        try:
            with open(file_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
                return text
        except Exception as e:
            logger.error(f"Error parsing PDF {file_path}: {str(e)}")
            return ""

    @staticmethod
    def parse_docx(file_path: str) -> str:
        """Extract text from DOCX"""
        try:
            doc = docx.Document(file_path)
            text = ""
            for paragraph in doc.paragraphs:
                text += paragraph.text + "\n"
            return text
        except Exception as e:
            logger.error(f"Error parsing DOCX {file_path}: {str(e)}")
            return ""

    @staticmethod
    def parse_txt(file_path: str) -> str:
        """Extract text from TXT"""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except Exception as e:
            logger.error(f"Error parsing TXT {file_path}: {str(e)}")
            return ""

class LegalKnowledgeBase:
    """Vector database for legal documents"""

    def __init__(self, storage_path: str = "legal_kb"):
        self.storage_path = Path(storage_path)
        self.storage_path.mkdir(exist_ok=True)

        self.documents: Dict[str, LegalDocument] = {}
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        self.document_vectors = None
        self.text_processor = LegalTextProcessor()

        # Load existing data
        self.load_knowledge_base()

    def add_document(self, document: LegalDocument) -> bool:
        """Add document to knowledge base"""
        try:
            # Process document content
            processed_content = self.text_processor.preprocess_legal_text(document.content)

            # Extract key terms
            document.key_terms = self.text_processor.extract_key_terms(document.content)

            # Generate summary
            document.summary = self._generate_summary(document.content)

            # Store document
            self.documents[document.id] = document

            # Update vectors
            self._update_vectors()

            # Save to disk
            self._save_document(document)

            logger.info(f"Added document: {document.title}")
            return True

        except Exception as e:
            logger.error(f"Error adding document: {str(e)}")
            return False

    def search_documents(self, query: LegalQuery, top_k: int = 5) -> List[Tuple[LegalDocument, float]]:
        """Search for relevant documents"""
        if not self.documents or self.document_vectors is None:
            return []

        try:
            # Process query
            processed_query = self.text_processor.preprocess_legal_text(query.query)
            query_vector = self.vectorizer.transform([processed_query])

            # Calculate similarities
            similarities = cosine_similarity(query_vector, self.document_vectors)[0]

            # Filter by practice area and jurisdiction if specified
            filtered_docs = []
            for doc_id, similarity in zip(self.documents.keys(), similarities):
                doc = self.documents[doc_id]

                # Apply filters
                if query.practice_area and query.practice_area not in doc.practice_area:
                    continue
                if query.jurisdiction and query.jurisdiction != doc.jurisdiction:
                    continue

                filtered_docs.append((doc, similarity))

            # Sort by similarity and return top_k
            filtered_docs.sort(key=lambda x: x[1], reverse=True)
            return filtered_docs[:top_k]

        except Exception as e:
            logger.error(f"Error searching documents: {str(e)}")
            return []

    def get_document_clusters(self, n_clusters: int = 5) -> Dict[int, List[str]]:
        """Cluster documents by similarity"""
        if len(self.documents) < n_clusters:
            return {}

        try:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            cluster_labels = kmeans.fit_predict(self.document_vectors)

            clusters = {}
            for doc_id, label in zip(self.documents.keys(), cluster_labels):
                if label not in clusters:
                    clusters[label] = []
                clusters[label].append(doc_id)

            return clusters

        except Exception as e:
            logger.error(f"Error clustering documents: {str(e)}")
            return {}

    def _update_vectors(self):
        """Update document vectors"""
        if not self.documents:
            return

        texts = []
        for doc in self.documents.values():
            processed_text = self.text_processor.preprocess_legal_text(doc.content)
            texts.append(processed_text)

        self.document_vectors = self.vectorizer.fit_transform(texts)

    def _generate_summary(self, content: str, max_sentences: int = 3) -> str:
        """Generate document summary"""
        try:
            sentences = sent_tokenize(content)
        except:
            # Fallback sentence splitting
            sentences = content.split('. ')

        if len(sentences) <= max_sentences:
            return content

        # Simple extractive summarization using TF-IDF
        try:
            vectorizer = TfidfVectorizer(stop_words='english')
            sentence_vectors = vectorizer.fit_transform(sentences)
            sentence_scores = np.sum(sentence_vectors.toarray(), axis=1)

            # Get top sentences
            top_indices = np.argsort(sentence_scores)[-max_sentences:]
            top_indices.sort()

            summary_sentences = [sentences[i] for i in top_indices]
            return ' '.join(summary_sentences)

        except:
            # Fallback to first few sentences
            return ' '.join(sentences[:max_sentences])

    def _save_document(self, document: LegalDocument):
        """Save document to disk"""
        doc_file = self.storage_path / f"{document.id}.json"
        with open(doc_file, 'w', encoding='utf-8') as f:
            json.dump(asdict(document), f, indent=2, ensure_ascii=False)

    def load_knowledge_base(self):
        """Load existing documents from disk"""
        if not self.storage_path.exists():
            return

        for doc_file in self.storage_path.glob("*.json"):
            try:
                with open(doc_file, 'r', encoding='utf-8') as f:
                    doc_data = json.load(f)
                    doc = LegalDocument(**doc_data)
                    self.documents[doc.id] = doc
            except Exception as e:
                logger.error(f"Error loading document {doc_file}: {str(e)}")

        if self.documents:
            self._update_vectors()
            logger.info(f"Loaded {len(self.documents)} documents")

class LegalRAGSystem:
    """Main Legal RAG system"""

    def __init__(self, kb_path: str = "legal_kb"):
        self.knowledge_base = LegalKnowledgeBase(kb_path)
        self.text_processor = LegalTextProcessor()
        self.document_parser = DocumentParser()

        # Legal templates
        self.response_templates = {
            'research': "Based on the legal documents, here's what I found regarding your research query:\n\n{content}\n\nKey legal considerations:\n{key_points}",
            'compliance': "Compliance Analysis:\n\n{content}\n\nRecommendations:\n{recommendations}",
            'drafting': "Document Drafting Guidance:\n\n{content}\n\nSuggested clauses:\n{clauses}",
            'analysis': "Legal Analysis:\n\n{content}\n\nRisk Assessment:\n{risks}"
        }

    def load_documents_from_directory(self, directory_path: str) -> int:
        """Load all legal documents from a directory"""
        directory = Path(directory_path)
        if not directory.exists():
            logger.error(f"Directory not found: {directory_path}")
            return 0

        loaded_count = 0
        for file_path in directory.rglob("*"):
            if file_path.is_file():
                if self.load_document(str(file_path)):
                    loaded_count += 1

        return loaded_count

    def load_document(self, file_path: str, document_type: str = "contract",
                     jurisdiction: str = "US", practice_areas: List[str] = None) -> bool:
        """Load a single document"""
        if practice_areas is None:
            practice_areas = ["general"]

        file_path = Path(file_path)
        if not file_path.exists():
            logger.error(f"File not found: {file_path}")
            return False

        # Parse document based on extension
        content = ""
        if file_path.suffix.lower() == '.pdf':
            content = self.document_parser.parse_pdf(str(file_path))
        elif file_path.suffix.lower() == '.docx':
            content = self.document_parser.parse_docx(str(file_path))
        elif file_path.suffix.lower() in ['.txt', '.md']:
            content = self.document_parser.parse_txt(str(file_path))
        else:
            logger.warning(f"Unsupported file format: {file_path.suffix}")
            return False

        if not content.strip():
            logger.warning(f"No content extracted from: {file_path}")
            return False

        # Create document object
        doc_id = hashlib.md5(f"{file_path.name}_{content[:100]}".encode()).hexdigest()
        document = LegalDocument(
            id=doc_id,
            title=file_path.stem,
            content=content,
            document_type=document_type,
            jurisdiction=jurisdiction,
            practice_area=practice_areas,
            date_created=datetime.now().isoformat(),
            key_terms=[]
        )

        return self.knowledge_base.add_document(document)

    def add_document_from_text(self, title: str, content: str,
                             document_type: str = "contract",
                             jurisdiction: str = "US",
                             practice_areas: List[str] = None) -> bool:
        """Add a document directly from text content"""
        if practice_areas is None:
            practice_areas = ["general"]

        if not content.strip():
            logger.warning("Empty content provided")
            return False

        # Create document object
        doc_id = hashlib.md5(f"{title}_{content[:100]}".encode()).hexdigest()
        document = LegalDocument(
            id=doc_id,
            title=title,
            content=content,
            document_type=document_type,
            jurisdiction=jurisdiction,
            practice_area=practice_areas,
            date_created=datetime.now().isoformat(),
            key_terms=[]
        )

        return self.knowledge_base.add_document(document)

    def query(self, query_text: str, practice_area: str = None,
              jurisdiction: str = None, query_type: str = "general") -> Dict[str, Any]:
        """Process legal query and return response"""

        # Create query object
        legal_query = LegalQuery(
            query=query_text,
            practice_area=practice_area,
            jurisdiction=jurisdiction,
            query_type=query_type
        )

        # Search for relevant documents
        relevant_docs = self.knowledge_base.search_documents(legal_query, top_k=5)

        if not relevant_docs:
            return {
                'response': "I couldn't find relevant legal documents for your query. Please try rephrasing or check if documents are loaded.",
                'sources': [],
                'confidence': 0.0,
                'legal_entities': {}
            }

        # Extract legal entities from query
        legal_entities = self.text_processor.extract_legal_entities(query_text)

        # Generate response
        response_content = self._generate_response(legal_query, relevant_docs)

        # Prepare sources
        sources = []
        for doc, similarity in relevant_docs:
            sources.append({
                'title': doc.title,
                'document_type': doc.document_type,
                'practice_area': doc.practice_area,
                'similarity': float(similarity),
                'key_terms': doc.key_terms,
                'summary': doc.summary
            })

        return {
            'response': response_content,
            'sources': sources,
            'confidence': float(relevant_docs[0][1]) if relevant_docs else 0.0,
            'legal_entities': legal_entities,
            'query_type': query_type
        }

    def _generate_response(self, query: LegalQuery, relevant_docs: List[Tuple[LegalDocument, float]]) -> str:
        """Generate response based on relevant documents"""

        # Combine content from relevant documents
        combined_content = []
        key_points = []

        for doc, similarity in relevant_docs:
            # Extract relevant sentences from each document
            try:
                sentences = sent_tokenize(doc.content)
            except:
                sentences = doc.content.split('. ')

            # Find sentences most relevant to query
            try:
                query_words = set(word_tokenize(query.query.lower()))
            except:
                query_words = set(query.query.lower().split())

            relevant_sentences = []

            for sentence in sentences[:20]:  # Limit for performance
                try:
                    sentence_words = set(word_tokenize(sentence.lower()))
                except:
                    sentence_words = set(sentence.lower().split())
                overlap = len(query_words.intersection(sentence_words))
                if overlap > 1:  # At least 2 words in common
                    relevant_sentences.append(sentence)

            if relevant_sentences:
                combined_content.extend(relevant_sentences[:3])  # Top 3 sentences per doc

            # Add key terms as key points
            key_points.extend(doc.key_terms[:3])

        # Format response based on query type
        template = self.response_templates.get(query.query_type, self.response_templates['research'])

        content_text = '\n'.join(combined_content[:10])  # Limit response length
        key_points_text = '\n'.join([f"• {point}" for point in set(key_points[:8])])

        if query.query_type == 'compliance':
            recommendations = self._generate_compliance_recommendations(combined_content)
            return template.format(content=content_text, recommendations=recommendations)
        elif query.query_type == 'drafting':
            clauses = self._generate_suggested_clauses(combined_content)
            return template.format(content=content_text, clauses=clauses)
        elif query.query_type == 'analysis':
            risks = self._generate_risk_assessment(combined_content)
            return template.format(content=content_text, risks=risks)
        else:
            return template.format(content=content_text, key_points=key_points_text)

    def _generate_compliance_recommendations(self, content: List[str]) -> str:
        """Generate compliance recommendations"""
        recommendations = [
            "• Review current policies and procedures",
            "• Ensure documentation is up to date",
            "• Consider regular compliance audits",
            "• Train relevant personnel on requirements"
        ]
        return '\n'.join(recommendations)

    def _generate_suggested_clauses(self, content: List[str]) -> str:
        """Generate suggested contract clauses"""
        clauses = [
            "• Include clear termination provisions",
            "• Define liability limitations",
            "• Specify governing law and jurisdiction",
            "• Add dispute resolution mechanisms"
        ]
        return '\n'.join(clauses)

    def _generate_risk_assessment(self, content: List[str]) -> str:
        """Generate risk assessment"""
        risks = [
            "• High: Review for regulatory compliance",
            "• Medium: Consider contractual protections",
            "• Low: Monitor industry developments",
            "• Ongoing: Maintain proper documentation"
        ]
        return '\n'.join(risks)

    def get_statistics(self) -> Dict[str, Any]:
        """Get system statistics"""
        docs = self.knowledge_base.documents

        stats = {
            'total_documents': len(docs),
            'document_types': {},
            'practice_areas': {},
            'jurisdictions': {},
            'total_terms': 0
        }

        for doc in docs.values():
            # Document types
            doc_type = doc.document_type
            stats['document_types'][doc_type] = stats['document_types'].get(doc_type, 0) + 1

            # Practice areas
            for area in doc.practice_area:
                stats['practice_areas'][area] = stats['practice_areas'].get(area, 0) + 1

            # Jurisdictions
            jurisdiction = doc.jurisdiction
            stats['jurisdictions'][jurisdiction] = stats['jurisdictions'].get(jurisdiction, 0) + 1

            # Terms
            stats['total_terms'] += len(doc.key_terms)

        return stats

# HOW TO USE THE SYSTEM - COMPREHENSIVE GUIDE
def usage_examples():
    """Comprehensive usage examples"""
    print("="*60)
    print("LEGAL RAG SYSTEM - USAGE GUIDE")
    print("="*60)

    # Initialize the system
    legal_rag = LegalRAGSystem()

    print("\n1. ADDING DOCUMENTS FROM FILES:")
    print("-" * 40)
    print("# Load a single document")
    print('legal_rag.load_document("path/to/contract.pdf", document_type="contract", jurisdiction="US", practice_areas=["corporate"])')

    print("\n# Load all documents from a directory")
    print('loaded_count = legal_rag.load_documents_from_directory("path/to/legal_docs/")')

    print("\n2. ADDING DOCUMENTS FROM TEXT:")
    print("-" * 40)
    print('# Add document directly from text')
    print('legal_rag.add_document_from_text(')
    print('    title="Privacy Policy Template",')
    print('    content="Your legal document text here...",')
    print('    document_type="policy",')
    print('    jurisdiction="US",')
    print('    practice_areas=["privacy", "compliance"]')
    print(')')

    print("\n3. QUERYING THE SYSTEM:")
    print("-" * 40)
    print('# General research query')
    print('result = legal_rag.query("What are the termination clauses in employment contracts?")')

    print('\n# Specific practice area query')
    print('result = legal_rag.query(')
    print('    "GDPR compliance requirements",')
    print('    practice_area="privacy",')
    print('    query_type="compliance"')
    print(')')

    print("\n4. QUERY TYPES:")
    print("-" * 40)
    print('• "research" - General legal research')
    print('• "compliance" - Compliance analysis with recommendations')
    print('• "drafting" - Document drafting guidance')
    print('• "analysis" - Legal analysis with risk assessment')

    print("\n5. SUPPORTED DOCUMENT TYPES:")
    print("-" * 40)
    print('• PDF files (.pdf)')
    print('• Word documents (.docx)')
    print('• Text files (.txt)')
    print('• Markdown files (.md)')

    return legal_rag

# Example usage and testing
def main():
    """Example usage of the Legal RAG system"""

    # Show usage guide first
    legal_rag = usage_examples()

    print("\n" + "="*60)
    print("RUNNING DEMO WITH SAMPLE DOCUMENTS")
    print("="*60)

    # Add sample documents using the new method
    sample_docs = [
        {
            'title': 'Software License Agreement',
            'content': '''
            This Software License Agreement ("Agreement") is entered into between Company A and User.
            The software is provided "as is" without warranty. User agrees to indemnify Company A against
            any claims arising from use of the software. This agreement is governed by the laws of California.
            Either party may terminate this agreement with 30 days written notice.
            The license fee is $1,000 per year. This agreement shall be effective from January 1, 2024.
            ''',
            'document_type': 'contract',
            'jurisdiction': 'US-CA',
            'practice_areas': ['intellectual_property', 'technology']
        },
        {
            'title': 'Employment Contract Template',
            'content': '''
            Employment Agreement between Employer and Employee. Employee agrees to work full-time
            and maintain confidentiality of proprietary information. Employer provides health benefits
            and 401k matching. Employment is at-will and may be terminated by either party.
            Non-compete clause applies for 1 year in same industry within 50 miles.
            Salary is $75,000 annually with performance bonuses available.
            ''',
            'document_type': 'contract',
            'jurisdiction': 'US',
            'practice_areas': ['employment', 'corporate']
        },
        {
            'title': 'Privacy Policy Requirements - GDPR',
            'content': '''
            Under GDPR regulations, companies must obtain explicit consent for data processing.
            Users have the right to access, modify, and delete their personal data.
            Data breaches must be reported within 72 hours to supervisory authorities.
            Companies must implement privacy by design and conduct impact assessments.
            Fines can reach 4% of annual global turnover for non-compliance.
            The regulation became effective on May 25, 2018 across all EU member states.
            ''',
            'document_type': 'regulation',
            'jurisdiction': 'EU',
            'practice_areas': ['privacy', 'data_protection', 'compliance']
        }
    ]

    # Add sample documents
    for doc_data in sample_docs:
        success = legal_rag.add_document_from_text(
            title=doc_data['title'],
            content=doc_data['content'],
            document_type=doc_data['document_type'],
            jurisdiction=doc_data['jurisdiction'],
            practice_areas=doc_data['practice_areas']
        )
        print(f"Added '{doc_data['title']}': {'✓' if success else '✗'}")

    print(f"\nLegal RAG System Initialized!")
    print(f"Loaded {len(legal_rag.knowledge_base.documents)} documents")

    # Example queries
    test_queries = [
        {
            'query': 'What are the requirements for software license termination?',
            'query_type': 'research',
            'practice_area': 'intellectual_property'
        },
        {
            'query': 'How do we ensure GDPR compliance for our website?',
            'query_type': 'compliance',
            'practice_area': 'privacy'
        },
        {
            'query': 'What should be included in an employment contract?',
            'query_type': 'drafting',
            'practice_area': 'employment'
        }
    ]

    print("\n" + "="*50)
    print("EXAMPLE QUERIES AND RESPONSES")
    print("="*50)

    for i, query_data in enumerate(test_queries, 1):
        print(f"\nQuery {i}: {query_data['query']}")
        print(f"Type: {query_data['query_type']}")
        print("-" * 40)

        result = legal_rag.query(
            query_data['query'],
            practice_area=query_data.get('practice_area'),
            query_type=query_data['query_type']
        )

        print("Response:")
        print(result['response'])
        print(f"\nConfidence: {result['confidence']:.2f}")
        print(f"Sources found: {len(result['sources'])}")

        if result['sources']:
            print("Top source:", result['sources'][0]['title'])

    # Show system statistics
    print("\n" + "="*50)
    print("SYSTEM STATISTICS")
    print("="*50)
    stats = legal_rag.get_statistics()
    for key, value in stats.items():
        print(f"{key}: {value}")

if __name__ == "__main__":
    main()

✓ punkt already available


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✓ punkt_tab downloaded successfully
✓ stopwords already available
✓ wordnet downloaded successfully


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✓ omw-1.4 downloaded successfully
LEGAL RAG SYSTEM - USAGE GUIDE

1. ADDING DOCUMENTS FROM FILES:
----------------------------------------
# Load a single document
legal_rag.load_document("path/to/contract.pdf", document_type="contract", jurisdiction="US", practice_areas=["corporate"])

# Load all documents from a directory
loaded_count = legal_rag.load_documents_from_directory("path/to/legal_docs/")

2. ADDING DOCUMENTS FROM TEXT:
----------------------------------------
# Add document directly from text
legal_rag.add_document_from_text(
    title="Privacy Policy Template",
    content="Your legal document text here...",
    document_type="policy",
    jurisdiction="US",
    practice_areas=["privacy", "compliance"]
)

3. QUERYING THE SYSTEM:
----------------------------------------
# General research query
result = legal_rag.query("What are the termination clauses in employment contracts?")

# Specific practice area query
result = legal_rag.query(
    "GDPR compliance requirements",
 